In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts.lightning_ssl_matched_speech_in_noise import LitAudioSSL
from jsinV3DataLoader_precombined_batched import CleanSpeechInNoiseValDatasetBatched

sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import pickle
from lightning_scripts.eval_jsin_transfer_matched import SSLClassifier

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

TASK = "word"
LAYER = "relu2" ## ckpt at this layerneeds to exist 

# config['data'] = {}
# config['data']['root'] = "/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/"
config['num_workers'] = 4
config['hparas']['batch_size'] = 32
config['data']['eval_max'] = 1
# config['hparas']['optimizer'] = args.optimizer
# config['hparas']['lr'] = args.lr * args.gpus
# config['hparas']['epochs'] = 2
# don't load in classifier head if it exists 
config['model']['arch_kwargs']['supervised'] =  False
config['model']['arch_kwargs']['time_average'] = False

if TASK == "word":
    config['data']['task_label'] = 'signal/word_int'
    config['data']['target_keys'] = ['signal/word_int']
    config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794} 
    task_str = f"word_task"

elif TASK == "speaker":
    config['data']['task_label'] = 'signal/speaker_int'
    config['model']['arch_kwargs']['n_classes'] =  433


config['hparas']['task_loss_params'] = {key:value for key,value in config['hparas']['task_loss_params'].items() if key in config['model']['arch_kwargs']['num_classes'].keys()}


In [3]:
ckpt_path = f"model_checkpoints/{config_path.stem}/linear_classifier_checkpoints_word_task_relu2_full_rep_AdamW_0.01_cosine_lr_scheduler_/epoch=0-step=28000.ckpt"
# ckpt_path = "model_checkpoints/resnet18_barlow_equivariant_lmbda_1e-2_lr_2e-1_no_avgpool_eq_lmbda_3e-01/linear_classifier_checkpoints_word_task_layer2_full_rep_AdamW_1e-05_w_dropout/epoch=2-step=66400.ckpt"
classifier_ckpt = torch.load(ckpt_path, weights_only=False) # get latest checkpoint 
# dummy init with checkpoint 
module = SSLClassifier(config=config, ckpt_path=ckpt_path, layer_out=LAYER)


module.load_state_dict(classifier_ckpt['state_dict'])

module = module.cuda()
## update keys to remove _orig_mod from eatch key 

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:191: Found keys that are in the model state dict but not in the checkpoint: ['audio_rep.rep.downsampling_op.downsample_filter', 'audio_rep.rep.Cochleagram.compute_subbands.coch_filters', 'audio_rep.rep.Cochleagram.downsampling.downsample_filter', 'model.front_end.rep.downsampling_op.downsample_filter', 'model.front_end.rep.Cochleagram.compute_subbands.coch_filters', 'model.front_end.rep.Cochleagram.downsampling.downsample_filter', 'model.model.f.batchnorm0.weight', 'model.model.f.batchnorm0.bias', 'model.model.f.batchnorm0.running_mean', 'model.model.f.batchnorm0.running_var', 'model.model.f.conv0.weight', 'model.model.f.batchnorm1.weight', 'model.model.f.batchnorm1.bias', 'model.model.f.batchnorm1.running_mean', 'model.model.f.batchnorm1.running_var', 'model.model.f.conv1.weight', 'model.model.f.batchnorm2.weight', 'model.model.f.batchnorm2.bias', 'model.model.f.batchnorm2.running_mea

In [4]:
# run test 


def collate_fn(batch):
    audio, targets = batch[0] # unbox wrapper added by dataloader 
    audio = audio.unsqueeze(1)
    # # combine labels: each target is dict for each key, stack the values 
    labels = {}
    for label_key in targets.keys():
        labels[label_key] = torch.from_numpy(targets[label_key])
    return audio, labels

test_dataset = CleanSpeechInNoiseValDatasetBatched(config['data']['speech_h5_path'],
                                            target_keys=config['data']['target_keys'],
                                            batch_size=100
)
test_dataset.target_keys = ['signal/word_int']
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=collate_fn
)

trainer = L.Trainer(     
        devices=1,
        accelerator="gpu",
        limit_predict_batches=500, 
)   

outputs = trainer.predict(module, test_dataloader, return_predictions=True)
top1_word = []
top5_word = []
n_examples = len(outputs)

for record in outputs:
    top1_word.append(record['top1']['signal/word_int'])
    top5_word.append(record['top5']['signal/word_int'])

output_dict = {
    "word_top1_mean": np.stack(top1_word).mean(),
    "word_top1_sem": np.stack(top1_word).std() / np.sqrt(n_examples),
    "word_top5_mean": np.stack(top5_word).mean(),
    "word_top5_sem": np.stack(top5_word).std() / np.sqrt(n_examples),
}
output_dict


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

{'word_top1_mean': np.float32(0.73456),
 'word_top1_sem': np.float64(0.002039806991410357),
 'word_top5_mean': np.float32(0.91977996),
 'word_top5_sem': np.float64(0.0018968138379854961)}

In [12]:
module.device

device(type='cpu')

In [ ]:

audio, labels = next(iter(test_dataloader))
word_labels = labels['signal/word_int']
module = module.cuda().eval()

with torch.no_grad():
    task_IXS = (word_labels != 0 ).nonzero(as_tuple=True)
    model_preds = module(audio.cuda())
    word_preds = model_preds['signal/word_int'].cpu().softmax(-1).cpu()[task_IXS]
    model_top_1 = word_preds.argmax(-1)
    word_labels = word_labels[task_IXS] 

## Cut audio to valid ixs for display
audio = audio[task_IXS]

In [14]:
raw_acc = (model_top_1 == word_labels).numpy().mean()
raw_acc

np.float64(0.75)

In [15]:
# model top5
top_5 = torch.isin(torch.topk(word_preds, k=5, dim=-1).indices, word_labels).any(-1).float().mean()
top_5

tensor(0.9200)

In [16]:
word_and_speaker_encodings = pickle.load(
    open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
)
class_map = word_and_speaker_encodings["word_idx_to_word"]

In [17]:
### Geck examples where model predicted wrong label 




model_failure_IXS = torch.where((model_top_1 != word_labels))[0].numpy()

for _ in range(20):

    failure_eg = int(model_failure_IXS[_])

    true_word = class_map[int(word_labels[failure_eg])]
    ## Get model top 1 and top 5 for that eg 
    model_pred = class_map[int(model_top_1[failure_eg])]

    # model top 5 transcripbed 
    model_eg_top5 = torch.topk(word_preds[failure_eg], k=5, dim=-1).indices
    model_eg_top5_words = [class_map[int(ix)] for ix in model_eg_top5] 

    print(f"True word: {true_word}")
    print(f"Model top 5 words: {', '.join(model_eg_top5_words)}")
    display(Audio(audio[failure_eg], rate=20_000, normalize=False))
    print("\n")


True word: eighty
Model top 5 words: about, after, eighty, enough, child




True word: three
Model top 5 words: estate, ability, above, accepted, __nullSignal__




True word: class
Model top 5 words: parents, class, above, ability, accepted




True word: which
Model top 5 words: works, ability, above, accepted, __nullSignal__




True word: people
Model top 5 words: company, ability, above, accepted, __nullSignal__


...



True word: company's
Model top 5 words: companies, company, above, ability, accepted




True word: district
Model top 5 words: cover, district, programs, release, project




True word: company
Model top 5 words: international, becoming, above, ability, accepted




True word: times
Model top 5 words: hundred, ability, above, accepted, __nullSignal__




True word: again
Model top 5 words: active, again, began, above, ability
